In [6]:
import os
import glob
import datetime
import numpy as np
import pandas as pd
import xarray as xr

### automatically refresh the buffer
%load_ext autoreload
%autoreload 2

### solve the auto-complete issue

%config Completer.use_jedi = False
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

### lvl 2 setups (systerm)
import os
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap,LinearSegmentedColormap,BoundaryNorm
import matplotlib.dates as mdates
import geopandas as gpd
from shapely.geometry import Point
from datetime import datetime


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Covariation of meteorological factors calculate

In [7]:
import xarray as xr
import numpy as np

def compute_RH_TEMP_VWS_LTS(ds):
    """
    Input
    -----
    ds : xarray.Dataset
        Required variables with dimensions (time, height) and their QC variables:
          rh, qc_rh,
          temp, qc_temp,
          u_wind, qc_u_wind,
          v_wind, qc_v_wind,
          potential_temp, qc_potential_temp

        The dataset must also contain a height coordinate in km.
        The height coordinate can be either ascending or descending.

    Output
    ------
    out : xarray.Dataset
        A dataset containing five time-series variables:
          RH_750_850      (%), height-weighted mean over 1.4567–2.4652 km
          TEMP_750_850    (K), same height range as RH
          VWS_725_925     (m s-1), u(2.7343 km) - u(0.7617 km)
          WSS_725_925     (m s-1), |V|(2.7343 km) - |V|(0.7617 km)
          LTS             (K), θ(3.0109 km) - θ(surface = height index 0)
    """
    # ---- Check required variables ----
    need = [
        "rh", "qc_rh",
        "temp", "qc_temp",
        "u_wind", "qc_u_wind",
        "v_wind", "qc_v_wind",
        "potential_temp", "qc_potential_temp",
    ]
    missing = [n for n in need if n not in ds]
    if missing:
        raise KeyError(f"Dataset is missing required variables: {missing}")
    if "height" not in ds.dims:
        raise KeyError("Dataset must contain the 'height' dimension in km.")

    # ---- Ensure height is in ascending order ----
    ds = ds.sortby("height")

    # ---- Apply QC filtering and keep only qc == 0 ----
    rh    = ds["rh"].where(ds["qc_rh"] == 0)
    T     = ds["temp"].where(ds["qc_temp"] == 0)
    u     = ds["u_wind"].where(ds["qc_u_wind"] == 0)
    v     = ds["v_wind"].where(ds["qc_v_wind"] == 0)
    theta = ds["potential_temp"].where(ds["qc_potential_temp"] == 0)

    # ---- Helper function: map target height to the nearest grid height ----
    def nearest_height(val_km: float) -> float:
        h = ds["height"].sel(height=val_km, method="nearest").item()
        return float(h)

    # ---- Helper function: height-weighted layer mean (∫v dh / ∫dh) ----
    def layer_mean_hweighted(da, hmin, hmax):
        """
        da : DataArray(time, height)

        Return
        ------
        DataArray(time,)

        Method
        ------
        Compute the layer mean over [hmin, hmax] using layer-thickness weights
        derived from the height-coordinate gradient.
        """
        h0 = nearest_height(hmin)
        h1 = nearest_height(hmax)
        if h0 > h1:
            h0, h1 = h1, h0

        sub = da.sel(height=slice(h0, h1))

        if sub.sizes.get("height", 0) == 0:
            return xr.full_like(da.isel(height=0), np.nan).drop_vars("height")

        h = sub["height"]
        dh = np.gradient(h.values)
        w = xr.DataArray(dh, coords={"height": h}, dims=("height",))

        num = (sub * w).sum(dim="height", skipna=True)
        den = w.where(sub.notnull()).sum(dim="height", skipna=True)

        out = num / den
        return out

    # ---- 1) Height-weighted RH/T mean over 1.4567–2.4652 km ----
    hmin, hmax = 1.4567, 2.4652

    RH_750_850 = layer_mean_hweighted(rh, hmin, hmax).rename("RH_750_850")
    RH_750_850.attrs.update({
        "units": "%",
        "long_name": "Height-weighted RH (≈750–850 hPa)"
    })

    TEMP_750_850 = layer_mean_hweighted(T, hmin, hmax).rename("TEMP_750_850")
    TEMP_750_850.attrs.update({
        "units": T.attrs.get("units", "K"),
        "long_name": "Height-weighted Temperature (≈750–850 hPa)"
    })

    # ---- 2) U-wind shear and wind-speed shear over the same 725–925 hPa height range ----
    h_725 = nearest_height(2.7343)  # ≈725 hPa
    h_925 = nearest_height(0.7617)  # ≈925 hPa

    u_725 = u.sel(height=h_725, method="nearest")
    u_925 = u.sel(height=h_925, method="nearest")
    v_725 = v.sel(height=h_725, method="nearest")
    v_925 = v.sel(height=h_925, method="nearest")

    # U-wind shear: u(725 hPa) - u(925 hPa)
    VWS_725_925 = (u_725 - u_925).rename("VWS_725_925")
    VWS_725_925.attrs.update({
        "units": "m s-1",
        "long_name": "Zonal wind shear u(≈725 hPa) - u(≈925 hPa)"
    })

    # Wind-speed shear: sqrt(u_725^2 + v_725^2) - sqrt(u_925^2 + v_925^2)
    wind_speed_725 = np.hypot(u_725, v_725)
    wind_speed_925 = np.hypot(u_925, v_925)

    WSS_725_925 = (wind_speed_725 - wind_speed_925).rename("WSS_725_925")
    WSS_725_925.attrs.update({
        "units": "m s-1",
        "long_name": "Wind-speed shear |V|(≈725 hPa) - |V|(≈925 hPa)"
    })

    # ---- 3) LTS = θ(3.0109 km) - θ(surface, height index == 0) ----
    h_700 = nearest_height(3.0109)  # ≈700 hPa

    theta_700 = theta.sel(height=h_700, method="nearest")
    theta_surface = theta.isel(height=0)

    LTS = (theta_700 - theta_surface).rename("LTS")
    LTS.attrs.update({
        "units": theta.attrs.get("units", "K"),
        "long_name": "Lower Tropospheric Stability (θ≈700 hPa − θ at surface)"
    })

    # ---- Merge output variables ----
    out = xr.merge([
        RH_750_850,
        TEMP_750_850,
        VWS_725_925,
        WSS_725_925,
        LTS,
    ])

    out.attrs["note"] = (
        "QC==0 used. RH/T are height-weighted means over ~750–850 hPa "
        "(endpoints snapped to nearest height). "
        "VWS_725_925 is zonal wind shear u(≈725)-u(≈925). "
        "WSS_725_925 is |V|(≈725)-|V|(≈925), where |V|=sqrt(u^2+v^2). "
        "LTS=θ(≈700)-θ(surface at height index 0)."
    )

    return out

In [9]:
# ===== Paths and parameters =====
base_dir = "/data/shared_data/ARM_data/ENA/others/enainterpolatedsondeC1.c1"
out_dir  = "/data/ggong/ARM_monthly/ENA/interpolatedsondeC1"
os.makedirs(out_dir, exist_ok=True)

vars_to_save = [
    "rh", "qc_rh",
    "u_wind", "qc_u_wind",
    "v_wind", "qc_v_wind",
    "temp", "qc_temp",
    "potential_temp", "qc_potential_temp",
]

# ===== Monthly sequence (modify the time range as needed) =====
months = pd.date_range("2016-01-01", "2025-12-01", freq="MS")


def _preprocess(ds):
    # Keep only variables that exist in ds to avoid errors caused by missing variables in some months
    keep = [v for v in vars_to_save if v in ds.variables]

    # Also keep coordinates and required auxiliary variables
    keep_coords = []
    for c in ["time", "height"]:
        if c in ds.coords or c in ds.variables:
            keep_coords.append(c)

    ds = ds[keep + keep_coords]

    return ds


# ===== Monthly processing loop =====
for m in months:
    ym = m.strftime("%Y%m")
    pattern = os.path.join(base_dir, f"enainterpolatedsondeC1.c1.{ym}*.nc")
    files = sorted(glob.glob(pattern))

    if not files:
        print(f"[SKIP] {ym}: no files found")
        continue

    try:
        # Read all files for the current month, automatically align by coordinates, and keep only required variables
        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            preprocess=_preprocess,
            parallel=True,
            decode_times=True,
            engine=None  # Let xarray automatically choose the backend engine
        )

        # Compute the five variables using the pre-defined function
        ds_out = compute_RH_TEMP_VWS_LTS(ds)

        # Define output filename and path
        out_name = f"interpolatedsondeC1_ENA_{ym}_height_average_wss.nc"
        out_path = os.path.join(out_dir, out_name)

        # Save directly without compression
        ds_out.to_netcdf(out_path)

        print(f"[OK] {ym}: output {out_path}")
        print(datetime.datetime.now())

    except Exception as e:
        print(f"[ERROR] {ym}: {e}")

    finally:
        # Close files promptly to release resources
        try:
            ds.close()
        except Exception:
            pass

        try:
            ds_out.close()
        except Exception:
            pass

[OK] 201601: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201601_height_average_wss.nc
[ERROR] 201601: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201602: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201602_height_average_wss.nc
[ERROR] 201602: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201603: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201603_height_average_wss.nc
[ERROR] 201603: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201604: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201604_height_average_wss.nc
[ERROR] 201604: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201605: output /data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_201605_height_average_wss.nc
[ERROR] 201605: type object 'datetime.datetime' has no attribute 'datetime'
[OK] 201606: ou

In [17]:
ds_1 = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/interpolatedsondeC1/interpolatedsondeC1_ENA_*_height_average_wss.nc')

In [18]:
START = "2016-01-01 00:00:00"
END   = "2025-12-31 23:59:00"

# Ensure time is in ascending order
ds_1 = ds_1.sortby("time")

# Target 2-minute time grid
target_time = pd.date_range(START, END, freq="2min")

# Linearly interpolate along the time dimension to 2-minute resolution
ds_2min = ds_1.interp(time=target_time, method="linear")

# Optional: check the output
print(ds_2min)
print(f"Number of new time points: {len(ds_2min.time)}")

<xarray.Dataset> Size: 74MB
Dimensions:       (time: 2630160)
Coordinates:
  * time          (time) datetime64[ns] 21MB 2016-01-01 ... 2025-12-31T23:58:00
Data variables:
    RH_750_850    (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
    TEMP_750_850  (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
    VWS_725_925   (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
    WSS_725_925   (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
    LTS           (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
Attributes:
    units:      %
    long_name:  Height-weighted RH (≈750–850 hPa)
    note:       QC==0 used. RH/T are height-weighted means over ~750–850 hPa ...
Number of new time points: 2630160


In [19]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/ENA/RH_T_VWS_LTS_2min_extend_wss.nc')

## calculate mete2 

In [ ]:
import xarray as xr
import numpy as np

def compute_SurT_SurWind_RH34_T34(ds):
    """
    Input
    -----
    ds : xarray.Dataset
        Required variables with dimensions (time, height) and their QC variables:
          temp, qc_temp,
          u_wind, qc_u_wind,
          v_wind, qc_v_wind,
          rh, qc_rh
        The dataset must also contain a height coordinate in km.
        The height coordinate can be either ascending or descending.

    Output
    ------
    out : xarray.Dataset
        A dataset containing four time-series variables:
          T_surf    (degC)   surface temperature at ds.height[0]
          T_34      (degC)   height-weighted mean temperature over 3–4 km
          Wind_surf (m s-1)  surface wind speed calculated from u/v
          RH_34     (%)      height-weighted mean RH over 3–4 km
    """
    # ---- Check required variables ----
    need = [
        "rh", "qc_rh",
        "temp", "qc_temp",
        "u_wind", "qc_u_wind",
        "v_wind", "qc_v_wind",
    ]
    missing = [n for n in need if n not in ds]
    if missing:
        raise KeyError(f"Dataset is missing required variables: {missing}")
    if "height" not in ds.dims:
        raise KeyError("Dataset must contain the 'height' dimension in km.")

    # ---- Ensure height is in ascending order ----
    ds = ds.sortby("height")

    # ---- Apply QC filtering and keep only qc == 0 ----
    rh = ds["rh"].where(ds["qc_rh"] == 0)
    T  = ds["temp"].where(ds["qc_temp"] == 0)
    u  = ds["u_wind"].where(ds["qc_u_wind"] == 0)
    v  = ds["v_wind"].where(ds["qc_v_wind"] == 0)

    # ---- Helper function: map target height to the nearest grid height ----
    def nearest_height(val_km: float) -> float:
        # sel(..., method="nearest") returns a DataArray; extract the scalar coordinate value
        h = ds["height"].sel(height=val_km, method="nearest").item()
        return float(h)

    # ---- Helper function: height-weighted layer mean (∫v dh / ∫dh) ----
    def layer_mean_hweighted(da, hmin, hmax):
        """
        da : DataArray(time, height)

        Return
        ------
        DataArray(time,)

        Method
        ------
        Compute the layer mean over [hmin, hmax] using layer-thickness weights
        derived from the height-coordinate gradient.
        """
        # Snap layer endpoints to the nearest available height levels
        h0 = nearest_height(hmin)
        h1 = nearest_height(hmax)
        if h0 > h1:
            h0, h1 = h1, h0

        sub = da.sel(height=slice(h0, h1))

        # If no height levels are selected, return all NaN along the time dimension
        if sub.sizes.get("height", 0) == 0:
            # da.isel(height=0) returns a DataArray with dimension (time,)
            return xr.full_like(da.isel(height=0), np.nan)

        # Use the geometric thickness of the original height grid as weights
        h = sub["height"]
        dh = np.gradient(h.values)  # 1D array
        w = xr.DataArray(dh, coords={"height": h}, dims=("height",))

        # Compute the valid-weight sum to avoid NaN propagation
        num = (sub * w).sum(dim="height", skipna=True)
        den = w.where(sub.notnull()).sum(dim="height", skipna=True)

        out = num / den
        return out

    # ---- 1) Height-weighted RH/T mean over 3–4 km ----
    hmin, hmax = 3, 4

    RH_34 = layer_mean_hweighted(rh, hmin, hmax).rename("RH_34")
    RH_34.attrs.update({
        "units": "percent",
        "long_name": "Height-weighted relative humidity (~3–4 km)",
        "layer_bounds_km": [hmin, hmax],
    })

    T_34 = layer_mean_hweighted(T, hmin, hmax).rename("T_34")
    T_34.attrs.update({
        "units": T.attrs.get("units", "degC"),
        "long_name": "Height-weighted temperature (~3–4 km)",
        "layer_bounds_km": [hmin, hmax],
    })

    # ---- 2) Surface wind speed (m/s) and surface temperature (degC) ----
    # Use height[0] as the lowest height level near the surface
    T_surf = T.isel(height=0).rename("T_surf")
    T_surf.attrs.update({
        "units": T.attrs.get("units", "degC"),
        "long_name": "Surface temperature (at lowest height level)",
        "note": "QC filtered: qc_temp == 0",
        "height_level_km": float(ds["height"].isel(height=0)),
    })

    # u and v have already been QC-filtered
    Wind_surf = np.hypot(
        u.isel(height=0),
        v.isel(height=0)
    ).rename("Wind_surf")
    Wind_surf.attrs.update({
        "units": "m s-1",
        "long_name": "Surface wind speed (from u, v at lowest height level)",
        "note": "QC filtered: qc_u_wind == 0 & qc_v_wind == 0",
        "height_level_km": float(ds["height"].isel(height=0)),
    })

    # ---- Merge output variables ----
    out = xr.merge([RH_34, T_34, Wind_surf, T_surf])
    out.attrs["note"] = (
        "Derived time series from profile dataset: "
        "QC filtering (qc_* == 0) applied to temp, rh, u_wind, v_wind. "
        "T_34 and RH_34 are geometrically height-weighted means over ~3–4 km, "
        "using layer thickness (Δheight) as weights. "
        "T_surf and Wind_surf are taken at the lowest height level."
    )

    return out

In [ ]:
# ===== Paths and parameters =====
base_dir = "/data/shared_data/ARM_data/ENA/others/interpolatedsondeC1.c1"
out_dir  = "/data/ggong/ARM_monthly/ENA/interpolatedsondeC1"
os.makedirs(out_dir, exist_ok=True)

vars_to_save = [
    "rh", "qc_rh",
    "temp", "qc_temp",
    "u_wind", "qc_u_wind",
    "v_wind", "qc_v_wind",
]

# ===== Monthly sequence; modify the time range as needed =====
months = pd.date_range("2024-01-01", "2025-12-01", freq="MS")


def _preprocess(ds):
    # Keep only variables that exist in ds to avoid errors caused by missing variables in some months
    keep = [v for v in vars_to_save if v in ds.variables]

    # Also keep coordinates and required auxiliary variables
    keep_coords = []
    for c in ["time", "height"]:
        if c in ds.coords or c in ds.variables:
            keep_coords.append(c)

    ds = ds[keep + keep_coords]

    return ds


# ===== Monthly processing loop =====
for m in months:
    ym = m.strftime("%Y%m")
    pattern = os.path.join(base_dir, f"enainterpolatedsondeC1.c1.{ym}*.nc")
    files = sorted(glob.glob(pattern))

    if not files:
        print(f"[SKIP] {ym}: no files found")
        continue

    try:
        # Read all files for the current month, automatically align by coordinates, and keep only required variables
        ds = xr.open_mfdataset(
            files,
            combine="by_coords",
            preprocess=_preprocess,
            parallel=True,
            decode_times=True,
        )

        # Compute the four variables using the pre-defined function
        ds_out = compute_SurT_SurWind_RH34_T34(ds)

        # Define output filename and path
        out_name = f"interpolatedsondeC1_ENA_{ym}_height_average_mete2.nc"
        out_path = os.path.join(out_dir, out_name)

        # Save directly without compression
        ds_out.to_netcdf(out_path)

        print(f"[OK] {ym}: output {out_path}")
        print(datetime.datetime.now())

    except Exception as e:
        print(f"[ERROR] {ym}: {e}")

    finally:
        # Close files promptly to release resources
        try:
            ds.close()
        except Exception:
            pass

        try:
            ds_out.close()
        except Exception:
            pass

In [22]:
ds_2 = xr.open_mfdataset('/data/ggong/ARM_monthly/ENA/interpolatedsondeC1/*mete2*')

In [ ]:
START = "2016-01-01 00:00:00"
END   = "2025-12-31 23:59:00"

# Ensure time is in ascending order
ds_2 = ds_2.sortby("time")

# Target 2-minute time grid
target_time = pd.date_range(START, END, freq="2min")

# Linearly interpolate along the time dimension to 2-minute resolution
ds_2min = ds_2.interp(time=target_time, method="linear")

# Optional: check the output
print(ds_2min)
print(f"Number of new time points: {len(ds_2min.time)}")

In [24]:
ds_2min

<xarray.Dataset> Size: 63MB
Dimensions:    (time: 2630160)
Coordinates:
    height     float32 4B 0.03048
  * time       (time) datetime64[ns] 21MB 2016-01-01 ... 2025-12-31T23:58:00
Data variables:
    RH_34      (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
    T_34       (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
    Wind_surf  (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
    T_surf     (time) float32 11MB dask.array<chunksize=(2630160,), meta=np.ndarray>
Attributes:
    units:            percent
    long_name:        Height-weighted relative humidity (~3–4 km)
    layer_bounds_km:  [3 4]
    note:             Derived time series from profile dataset: QC filtering ...

In [25]:
ds_2min.to_netcdf('/data/ggong/ARM_monthly/ENA/RH34_T34_Tsurf_Windsurf_2min_extend.nc')

## LTS = Cloud height + 2 resolution 

In [ ]:
import xarray as xr
import numpy as np

def compute_cloud_top_LTS(ds_lts, ds_ctop,
                          theta_name="potential_temp",
                          ctop_name="cloud_top_height"):
    """
    Compute LTS = θ(first height level above cloud top + 0.4 km) - θ(surface, height index 0)

    Parameters
    ----------
    ds_lts : xarray.Dataset
        Dataset containing potential temperature with dimensions (time, height), for example:
        - potential_temp(time, height)
        - height(height)   # unit: km

    ds_ctop : xarray.Dataset
        Dataset containing cloud-top height time series:
        - cloud_top_height(time)  # unit: km

    Returns
    -------
    LTS : xarray.DataArray  (time,)
    """

    # ---- Merge the two datasets and align them by time ----
    ds = xr.merge(
        [ds_lts[[theta_name]], ds_ctop[[ctop_name]]],
        join="inner"
    )

    # ---- Ensure height is in ascending order ----
    ds = ds.sortby("height")

    theta = ds[theta_name]      # (time, height)
    ctop  = ds[ctop_name]       # (time,)

    # ===== Key modification: use cloud_top + 0.4 km as the threshold =====
    ctop_plus = ctop + 0.4      # (time,)

    # Broadcast height and cloud_top+0.4 km to shape (time, height)
    H, C = xr.broadcast(ds["height"], ctop_plus)   # H, C: (time, height)

    # Mask all height levels above cloud_top + 0.4 km
    mask = H > C                                # (time, height)

    # Keep only the first True level
    first_mask = mask & (mask.cumsum("height") == 1)

    # Extract θ at the selected grid points; all other heights become NaN
    theta_first_above = theta.where(first_mask)

    # Only one valid height remains for each time, so max over height retrieves θ at that level
    theta_ctop_plus = theta_first_above.max("height", skipna=True)

    # Near-surface θ at height index 0
    theta_surface = theta.isel(height=0)

    LTS = (theta_ctop_plus - theta_surface).rename("LTS")
    LTS.attrs.update({
        "units": theta.attrs.get("units", "K"),
        "long_name": "Lower Tropospheric Stability (theta at first level above cloud top+0.4 km minus surface)"
    })

    return LTS

In [9]:
LTS = compute_cloud_top_LTS(ds_lts, ds_ctop)

In [11]:
LTS.to_netcdf('/data/ggong/ARM_monthly/ENA/LTS_cloud_top_plus_2_extend.nc')